# ML Design Patterns
### Five patterns that recur constantly in production ML systems

| # | Pattern | ML Use Case |
|---|---------|-------------|
| 1 | **Strategy** | Swap loss functions, optimizers, schedulers without changing the Trainer |
| 2 | **Factory / Registry** | Map string config names to class constructors |
| 3 | **Observer / Callback** | Decoupled side-effects (logging, checkpointing, early stopping) |
| 4 | **Template Method** | Define skeleton training loop in base class; subclasses override steps |
| 5 | **Adapter** | Wrap third-party APIs (HuggingFace, ONNX Runtime) behind stable internal interfaces |

> Each section: motivation → annotated implementation → live demo you can run.

In [1]:
# ── shared imports used across all patterns ──────────────────────────────
import math, time, random, copy
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Protocol, Type
import numpy as np

# Lightweight stand-ins so the notebook runs without a GPU / full PyTorch
class _FakeTensor:
    """Minimal tensor-like for demonstration without a GPU requirement."""
    def __init__(self, data):
        self.data = np.asarray(data, dtype=np.float32)
    def __repr__(self):  return f"Tensor({self.data})"
    def backward(self): pass
    def item(self):     return float(self.data.mean())
    def __add__(self, o): return _FakeTensor(self.data + (o.data if isinstance(o,_FakeTensor) else o))
    def __mul__(self, s): return _FakeTensor(self.data * s)
    def __truediv__(self, s): return _FakeTensor(self.data / s)

def fake_tensor(*shape): return _FakeTensor(np.random.randn(*shape))

print("✓ imports ready")

✓ imports ready


---
## Pattern 1 — Strategy

**Intent:** Define a *family of algorithms*, encapsulate each one, and make them interchangeable.
The client (Trainer) holds a reference to a strategy object and delegates the algorithm to it.

**ML Problem it solves:**  
You don't want an `if loss_type == 'focal': ... elif loss_type == 'mse': ...` chain inside your training loop. That violates Open/Closed and makes the loop impossible to test in isolation.

```
Trainer ──────► LossFn (Protocol/ABC)
                 ├── FocalLoss
                 ├── MSELoss
                 └── DiceLoss
```

In [2]:
# ── 1a: Define the Strategy interface via Protocol ────────────────────────
# Using Protocol means ANY callable with this signature satisfies the contract
# — no forced inheritance required.

class LossFn(Protocol):
    """Strategy interface for all loss functions."""
    def __call__(self, preds: _FakeTensor, targets: _FakeTensor) -> _FakeTensor: ...


# ── 1b: Concrete strategies ───────────────────────────────────────────────

class MSELoss:
    """Mean squared error — regression tasks."""
    def __call__(self, preds, targets):
        diff = preds.data - targets.data
        return _FakeTensor(np.mean(diff ** 2))


class FocalLoss:
    """
    Focal loss — down-weights easy negatives for class-imbalanced detection.
    L_fl = -alpha * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25):
        self.gamma = gamma
        self.alpha = alpha

    def __call__(self, preds, targets):
        # Simplified sigmoid focal loss for demo
        p      = 1 / (1 + np.exp(-preds.data))          # sigmoid
        p_t    = np.where(targets.data == 1, p, 1 - p)
        fl     = -self.alpha * (1 - p_t) ** self.gamma * np.log(np.clip(p_t, 1e-8, 1))
        return _FakeTensor(np.mean(fl))


class DiceLoss:
    """Dice loss — common in segmentation; handles class imbalance differently to focal."""
    def __init__(self, smooth: float = 1.0):
        self.smooth = smooth

    def __call__(self, preds, targets):
        p = 1 / (1 + np.exp(-preds.data))
        intersection = (p * targets.data).sum()
        dice = (2 * intersection + self.smooth) / (p.sum() + targets.data.sum() + self.smooth)
        return _FakeTensor(1 - dice)


# ── 1c: The Trainer holds a LossFn reference — knows nothing about its internals

class Trainer:
    """
    Trainer is the Context in Strategy terminology.
    It delegates loss computation entirely to whatever strategy was injected.
    """
    def __init__(self, loss_fn: LossFn):
        self.loss_fn  = loss_fn
        self.history: List[float] = []

    def train_step(self, preds: _FakeTensor, targets: _FakeTensor) -> float:
        loss = self.loss_fn(preds, targets)
        # loss.backward()  # would happen here in real PyTorch
        value = loss.item()
        self.history.append(value)
        return value

    # KEY: swap strategy at runtime — no Trainer code changes needed
    def set_loss(self, loss_fn: LossFn) -> None:
        self.loss_fn = loss_fn
        print(f"  ↳ loss strategy swapped to {loss_fn.__class__.__name__}")


print("Strategy classes defined ✓")

Strategy classes defined ✓


In [3]:
# ── Demo: run three steps with each strategy, then swap mid-run ──────────

np.random.seed(42)
preds   = fake_tensor(8)   # 8 predictions
targets = fake_tensor(8)

print("=== Strategy Pattern Demo ===")

for strategy_cls, kwargs in [
    (MSELoss,   {}),
    (FocalLoss, {"gamma": 2.0, "alpha": 0.25}),
    (DiceLoss,  {"smooth": 1.0}),
]:
    trainer = Trainer(loss_fn=strategy_cls(**kwargs))
    losses  = [trainer.train_step(fake_tensor(8), targets) for _ in range(3)]
    print(f"  {strategy_cls.__name__:12s} → losses: {[f'{v:.4f}' for v in losses]}")

print()
print("--- Runtime strategy swap ---")
trainer = Trainer(loss_fn=MSELoss())
print(f"  Before swap: {trainer.train_step(preds, targets):.4f}")
trainer.set_loss(FocalLoss(gamma=1.5))
print(f"  After  swap: {trainer.train_step(preds, targets):.4f}")

=== Strategy Pattern Demo ===
  MSELoss      → losses: ['1.2179', '1.4755', '0.7162']
  FocalLoss    → losses: ['0.0581', '0.0795', '0.0418']
  DiceLoss     → losses: ['4.1680', '-17.1096', '12.0807']

--- Runtime strategy swap ---
  Before swap: 2.7898
  ↳ loss strategy swapped to FocalLoss
  After  swap: 0.1515


In [4]:
# ── 1d: Strategy also applies to optimizers and LR schedulers ─────────────

class LRScheduler(Protocol):
    """Strategy interface for learning-rate schedules."""
    def get_lr(self, step: int, base_lr: float) -> float: ...


class CosineAnnealingScheduler:
    def __init__(self, T_max: int):
        self.T_max = T_max
    def get_lr(self, step, base_lr):
        return base_lr * 0.5 * (1 + math.cos(math.pi * step / self.T_max))


class LinearWarmupScheduler:
    def __init__(self, warmup_steps: int, T_max: int):
        self.warmup_steps = warmup_steps
        self.T_max        = T_max
    def get_lr(self, step, base_lr):
        if step < self.warmup_steps:
            return base_lr * step / max(1, self.warmup_steps)
        progress = (step - self.warmup_steps) / max(1, self.T_max - self.warmup_steps)
        return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


print("LR Scheduler strategies:")
BASE_LR, STEPS = 1e-3, 10
for sched in [CosineAnnealingScheduler(STEPS), LinearWarmupScheduler(3, STEPS)]:
    lrs = [f"{sched.get_lr(s, BASE_LR):.5f}" for s in range(STEPS)]
    print(f"  {sched.__class__.__name__:28s}: {lrs}")

LR Scheduler strategies:
  CosineAnnealingScheduler    : ['0.00100', '0.00098', '0.00090', '0.00079', '0.00065', '0.00050', '0.00035', '0.00021', '0.00010', '0.00002']
  LinearWarmupScheduler       : ['0.00000', '0.00033', '0.00067', '0.00100', '0.00095', '0.00081', '0.00061', '0.00039', '0.00019', '0.00005']


---
## Pattern 2 — Factory / Registry

**Intent:** Define an interface for creating objects, but let subclasses (or a registry) decide which class to instantiate.

**ML Problem it solves:**  
Config-driven experiments need to instantiate models by name (`model: rf_detr`) without
the orchestration code importing every model file. The registry maps string keys → classes
and eliminates if/elif chains entirely.

```
config.yaml                        Registry
──────────                         ────────
model: rf_detr        ──────►   { "rf_detr":     RFDETRModel,
backbone: dinov2_vitb           "phoenix_detr": PhoenixDETR,
                                  "yolov8":       YOLOv8 }
                                       │
                                       ▼  build_model(cfg)
                                    RFDETRModel(backbone="dinov2_vitb")
```

In [5]:
# ── 2a: The Registry ──────────────────────────────────────────────────────

class Registry:
    """
    Generic registry that maps string keys to classes.
    Works as a class decorator: @model_registry.register("rf_detr")
    """
    def __init__(self, name: str):
        self._name  = name
        self._store: Dict[str, Type] = {}

    def register(self, key: str) -> Callable:
        """Decorator that adds the class to the registry under `key`."""
        def decorator(cls):
            if key in self._store:
                raise KeyError(f"[{self._name}] duplicate key '{key}'")
            self._store[key] = cls
            return cls          # return unchanged so the class is still usable directly
        return decorator

    def build(self, key: str, **kwargs) -> Any:
        """Instantiate a registered class by name."""
        if key not in self._store:
            available = list(self._store)
            raise KeyError(f"[{self._name}] unknown key '{key}'. Available: {available}")
        return self._store[key](**kwargs)

    def keys(self) -> List[str]:
        return list(self._store)

    def __repr__(self):
        return f"Registry('{self._name}', keys={self.keys()})"


# ── 2b: Separate registries for separate component families ───────────────
model_registry    = Registry("models")
backbone_registry = Registry("backbones")
loss_registry     = Registry("losses")

print("Registries created ✓")

Registries created ✓


In [6]:
# ── 2c: Register concrete classes via decorator ───────────────────────────

class DetectionModel(ABC):
    """Abstract base for all detection models."""
    @abstractmethod
    def forward(self, images: list) -> list: ...
    @abstractmethod
    def param_count(self) -> int: ...


@model_registry.register("rf_detr")
class RFDETRModel(DetectionModel):
    def __init__(self, backbone: str = "dinov2_vitb14", num_classes: int = 10):
        self.backbone    = backbone
        self.num_classes = num_classes
    def forward(self, images):
        return [{"boxes": fake_tensor(5, 4), "scores": fake_tensor(5)} for _ in images]
    def param_count(self):
        return 86_000_000
    def __repr__(self):
        return f"RFDETRModel(backbone={self.backbone}, classes={self.num_classes})"


@model_registry.register("phoenix_detr")
class PhoenixDETR(DetectionModel):
    def __init__(self, backbone: str = "dinov2_vitl14", num_classes: int = 10):
        self.backbone    = backbone
        self.num_classes = num_classes
    def forward(self, images):
        return [{"boxes": fake_tensor(8, 4), "scores": fake_tensor(8)} for _ in images]
    def param_count(self):
        return 307_000_000
    def __repr__(self):
        return f"PhoenixDETR(backbone={self.backbone}, classes={self.num_classes})"


@model_registry.register("yolov8_nano")
class YOLOv8Nano(DetectionModel):
    def __init__(self, num_classes: int = 10):
        self.num_classes = num_classes
    def forward(self, images):
        return [{"boxes": fake_tensor(12, 4), "scores": fake_tensor(12)} for _ in images]
    def param_count(self):
        return 3_200_000
    def __repr__(self):
        return f"YOLOv8Nano(classes={self.num_classes})"


@loss_registry.register("focal")
class FocalLossV2(FocalLoss): pass  # reuse from Pattern 1

@loss_registry.register("mse")
class MSELossV2(MSELoss): pass


print("Registered:", model_registry)
print("Registered:", loss_registry)

Registered: Registry('models', keys=['rf_detr', 'phoenix_detr', 'yolov8_nano'])
Registered: Registry('losses', keys=['focal', 'mse'])


In [7]:
# ── 2d: build_from_config — the single entrypoint used in training scripts

def build_from_config(cfg: dict) -> DetectionModel:
    """
    Reads a config dict (as loaded from config.yaml) and builds the model.
    No knowledge of concrete classes required here.
    """
    model_key = cfg["model"]
    kwargs    = cfg.get("model_kwargs", {})
    model     = model_registry.build(model_key, **kwargs)
    return model


print("=== Factory / Registry Demo ===")
configs = [
    {"model": "rf_detr",     "model_kwargs": {"backbone": "dinov2_vitb14", "num_classes": 5}},
    {"model": "phoenix_detr","model_kwargs": {"backbone": "dinov2_vitl14", "num_classes": 5}},
    {"model": "yolov8_nano", "model_kwargs": {"num_classes": 5}},
]

for cfg in configs:
    m = build_from_config(cfg)
    preds = m.forward(["img1", "img2"])
    print(f"  {m}")
    print(f"    params: {m.param_count():>12,}   preds per image: {len(preds[0]['boxes'].data)}")

print()
print("--- Error on unknown key ---")
try:
    model_registry.build("deformable_detr")
except KeyError as e:
    print(f"  KeyError: {e}")

=== Factory / Registry Demo ===
  RFDETRModel(backbone=dinov2_vitb14, classes=5)
    params:   86,000,000   preds per image: 5
  PhoenixDETR(backbone=dinov2_vitl14, classes=5)
    params:  307,000,000   preds per image: 8
  YOLOv8Nano(classes=5)
    params:    3,200,000   preds per image: 12

--- Error on unknown key ---
  KeyError: "[models] unknown key 'deformable_detr'. Available: ['rf_detr', 'phoenix_detr', 'yolov8_nano']"


---
## Pattern 3 — Observer / Callback

**Intent:** Define a one-to-many dependency so that when the *Subject* (Trainer) changes state,
all registered *Observers* (Callbacks) are notified automatically.

**ML Problem it solves:**  
Checkpointing, early stopping, TensorBoard logging, and learning-rate scheduling all need
to "hook in" to the training loop at specific moments — but they are independent concerns.
Embedding them in the Trainer violates SRP. Callbacks decouple them cleanly.

```
Trainer.fit()
  on_epoch_end(metrics)  ──────►  [CheckpointCallback, EarlyStopCallback, LRSchedulerCallback]
  on_train_end()         ──────►  [LogCallback]
```

In [8]:
# ── 3a: Callback base class (the Observer interface) ─────────────────────

@dataclass
class TrainState:
    """Shared mutable state passed to every callback."""
    epoch:       int   = 0
    global_step: int   = 0
    train_loss:  float = 0.0
    val_map:     float = 0.0
    lr:          float = 1e-4
    stop_training: bool = False          # early stopping sets this to True
    best_val_map:  float = 0.0
    checkpoints:   List[str] = field(default_factory=list)


class Callback(ABC):
    """Base observer. Override only the hooks you care about."""
    def on_train_begin(self, state: TrainState) -> None:  pass
    def on_epoch_begin(self, state: TrainState) -> None:  pass
    def on_batch_end  (self, state: TrainState) -> None:  pass
    def on_epoch_end  (self, state: TrainState) -> None:  pass
    def on_train_end  (self, state: TrainState) -> None:  pass


print("Callback interface defined ✓")

Callback interface defined ✓


In [9]:
# ── 3b: Concrete callback implementations ────────────────────────────────

class CheckpointCallback(Callback):
    """
    Saves a checkpoint whenever val_map improves.
    In real code this would call torch.save(); here we log the event.
    """
    def __init__(self, save_dir: str = "./checkpoints"):
        self.save_dir = save_dir

    def on_epoch_end(self, state: TrainState) -> None:
        if state.val_map > state.best_val_map:
            state.best_val_map = state.val_map
            path = f"{self.save_dir}/epoch{state.epoch:03d}_map{state.val_map:.4f}.pth"
            state.checkpoints.append(path)
            print(f"  [Checkpoint] ✓ saved → {path}")


class EarlyStoppingCallback(Callback):
    """
    Halts training if val_map has not improved for `patience` epochs.
    Sets state.stop_training = True — Trainer checks this flag.
    """
    def __init__(self, patience: int = 3, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self._wait      = 0
        self._best      = -float("inf")

    def on_epoch_end(self, state: TrainState) -> None:
        if state.val_map > self._best + self.min_delta:
            self._best = state.val_map
            self._wait = 0
        else:
            self._wait += 1
            print(f"  [EarlyStopping] no improvement for {self._wait}/{self.patience} epochs")
            if self._wait >= self.patience:
                state.stop_training = True
                print(f"  [EarlyStopping] ✗ stopping training at epoch {state.epoch}")


class LRSchedulerCallback(Callback):
    """
    Applies a cosine-annealing LR schedule by updating state.lr each epoch.
    """
    def __init__(self, base_lr: float, T_max: int):
        self.base_lr = base_lr
        self.T_max   = T_max

    def on_epoch_end(self, state: TrainState) -> None:
        state.lr = self.base_lr * 0.5 * (1 + math.cos(math.pi * state.epoch / self.T_max))


class MetricsLoggerCallback(Callback):
    """Collects metrics into a history list for later plotting."""
    def __init__(self):
        self.history: List[dict] = []

    def on_epoch_end(self, state: TrainState) -> None:
        self.history.append({
            "epoch":      state.epoch,
            "train_loss": state.train_loss,
            "val_map":    state.val_map,
            "lr":         state.lr,
        })

    def on_train_end(self, state: TrainState) -> None:
        print(f"  [Logger] training complete — {len(self.history)} epochs logged")
        print(f"           best val_map: {max(h['val_map'] for h in self.history):.4f}")


print("Concrete callbacks defined ✓")

Concrete callbacks defined ✓


In [10]:
# ── 3c: CallbackTrainer — the Subject that fires events ──────────────────

class CallbackTrainer:
    """
    Trainer that knows nothing about checkpointing, early-stopping, or logging.
    All those concerns live in the registered Callback objects.
    """
    def __init__(self, callbacks: Optional[List[Callback]] = None):
        self.callbacks: List[Callback] = callbacks or []

    def _fire(self, event: str, state: TrainState) -> None:
        for cb in self.callbacks:
            getattr(cb, event)(state)

    def fit(self, epochs: int) -> TrainState:
        state = TrainState(lr=1e-3)
        self._fire("on_train_begin", state)

        for epoch in range(1, epochs + 1):
            state.epoch = epoch
            self._fire("on_epoch_begin", state)

            # ── Simulate a training epoch ─────────────────────────────────
            state.train_loss = max(0.05, 1.0 - epoch * 0.09 + random.gauss(0, 0.02))
            # Simulate val_map that plateaus mid-training to trigger early stop
            if epoch <= 6:
                state.val_map = min(0.95, 0.3 + epoch * 0.1 + random.gauss(0, 0.01))
            else:
                state.val_map = 0.88 + random.gauss(0, 0.005)   # plateau

            print(f"\nEpoch {epoch:02d} | loss={state.train_loss:.4f}  mAP={state.val_map:.4f}  lr={state.lr:.6f}")
            self._fire("on_epoch_end", state)

            if state.stop_training:
                break

        self._fire("on_train_end", state)
        return state


# ── Demo ───────────────────────────────────────────────────────────────────
print("=== Observer / Callback Demo ===")
random.seed(7)

logger     = MetricsLoggerCallback()
trainer    = CallbackTrainer(callbacks=[
    LRSchedulerCallback(base_lr=1e-3, T_max=12),
    CheckpointCallback(save_dir="./ckpts"),
    EarlyStoppingCallback(patience=3),
    logger,
])

final_state = trainer.fit(epochs=15)
print(f"\nSaved checkpoints: {final_state.checkpoints}")

=== Observer / Callback Demo ===

Epoch 01 | loss=0.9049  mAP=0.4051  lr=0.001000
  [Checkpoint] ✓ saved → ./ckpts/epoch001_map0.4051.pth

Epoch 02 | loss=0.8155  mAP=0.4968  lr=0.000983
  [Checkpoint] ✓ saved → ./ckpts/epoch002_map0.4968.pth

Epoch 03 | loss=0.7114  mAP=0.5979  lr=0.000933
  [Checkpoint] ✓ saved → ./ckpts/epoch003_map0.5979.pth

Epoch 04 | loss=0.6622  mAP=0.7042  lr=0.000854
  [Checkpoint] ✓ saved → ./ckpts/epoch004_map0.7042.pth

Epoch 05 | loss=0.5707  mAP=0.8025  lr=0.000750
  [Checkpoint] ✓ saved → ./ckpts/epoch005_map0.8025.pth

Epoch 06 | loss=0.4679  mAP=0.9019  lr=0.000629
  [Checkpoint] ✓ saved → ./ckpts/epoch006_map0.9019.pth

Epoch 07 | loss=0.3367  mAP=0.8843  lr=0.000500
  [EarlyStopping] no improvement for 1/3 epochs

Epoch 08 | loss=0.2901  mAP=0.8825  lr=0.000371
  [EarlyStopping] no improvement for 2/3 epochs

Epoch 09 | loss=0.1562  mAP=0.8713  lr=0.000250
  [EarlyStopping] no improvement for 3/3 epochs
  [EarlyStopping] ✗ stopping training at epoch

---
## Pattern 4 — Template Method

**Intent:** Define the *skeleton* of an algorithm in a base class and let subclasses
override specific *steps* without changing the overall structure.

**ML Problem it solves:**  
A standard training loop (`fit → train_epoch → val_epoch → log`) is the same for
classification, detection, and segmentation. Only the forward pass, loss, and metrics differ.
The Template Method fixes the loop structure while allowing customisation of each step.

```
BaseTrainer.fit()                   ← TEMPLATE METHOD (do not override)
  ├── setup()                       ← hook (optional override)
  ├── train_epoch()                 ← hook (optional override)
  │     └── train_step()            ← ABSTRACT — must override
  ├── val_epoch()                   ← hook (optional override)
  │     └── val_step()              ← ABSTRACT — must override
  └── teardown()                    ← hook (optional override)
```

In [11]:
# ── 4a: BaseTrainer — the Template Method ────────────────────────────────

@dataclass
class EpochMetrics:
    loss:    float = 0.0
    val_map: float = 0.0


class BaseTrainer(ABC):
    """
    Defines the invariant training loop skeleton.
    Subclasses implement train_step and val_step; everything else is inherited.
    """

    def fit(self, n_epochs: int) -> List[EpochMetrics]:
        """Template method — do NOT override this."""
        history = []
        self.setup()
        for epoch in range(1, n_epochs + 1):
            m = EpochMetrics()
            m.loss    = self.train_epoch(epoch)
            m.val_map = self.val_epoch(epoch)
            self._log(epoch, m)
            history.append(m)
        self.teardown()
        return history

    # ── Steps subclasses MUST implement ──────────────────────────────────
    @abstractmethod
    def train_step(self, batch: dict) -> float:
        """Run one batch forward+backward pass. Returns scalar loss."""
        ...

    @abstractmethod
    def val_step(self, batch: dict) -> float:
        """Run one validation batch. Returns mAP contribution."""
        ...

    # ── Hooks subclasses MAY override ────────────────────────────────────
    def setup(self):           pass
    def teardown(self):        pass

    def train_epoch(self, epoch: int) -> float:
        """Default: average loss over N fake batches. Override for custom looping."""
        losses = [self.train_step({"images": fake_tensor(4, 3, 512, 512)}) for _ in range(5)]
        return float(np.mean(losses))

    def val_epoch(self, epoch: int) -> float:
        maps = [self.val_step({"images": fake_tensor(4, 3, 512, 512)}) for _ in range(3)]
        return float(np.mean(maps))

    def _log(self, epoch, m: EpochMetrics):
        print(f"  epoch {epoch:02d} | loss={m.loss:.4f}  mAP={m.val_map:.4f}")


print("BaseTrainer skeleton defined ✓")

BaseTrainer skeleton defined ✓


In [12]:
# ── 4b: Concrete subclasses — only override what changes ─────────────────

class DetectionTrainer(BaseTrainer):
    """
    Detection-specific trainer.
    Overrides: train_step (focal + box regression), val_step (mAP computation).
    Inherits: fit(), train_epoch(), val_epoch() unchanged.
    """
    def __init__(self, model: DetectionModel):
        self.model    = model
        self.loss_fn  = FocalLoss(gamma=2.0)

    def setup(self):
        print(f"  [DetectionTrainer] setup — model: {self.model}")

    def train_step(self, batch: dict) -> float:
        preds  = self.model.forward(batch["images"].data.tolist())
        # Simulate combined focal + L1 regression loss
        cls_loss = self.loss_fn(fake_tensor(10), fake_tensor(10)).item()
        reg_loss = float(np.random.uniform(0.05, 0.3))
        return cls_loss + 0.5 * reg_loss

    def val_step(self, batch: dict) -> float:
        # Simulate mAP@50
        return float(np.random.uniform(0.55, 0.75))


class SegmentationTrainer(BaseTrainer):
    """
    Segmentation-specific trainer.
    Uses Dice loss; val_step returns mIoU instead of mAP.
    The fit() skeleton is identical — only the steps differ.
    """
    def __init__(self):
        self.loss_fn = DiceLoss(smooth=1.0)

    def setup(self):
        print("  [SegmentationTrainer] setup — Dice loss, mIoU metric")

    def train_step(self, batch: dict) -> float:
        preds   = fake_tensor(4, 1, 512, 512)
        targets = fake_tensor(4, 1, 512, 512)
        return self.loss_fn(preds, targets).item()

    def val_step(self, batch: dict) -> float:
        # Returns mIoU — same slot, different semantics
        return float(np.random.uniform(0.60, 0.80))

    def _log(self, epoch, m: EpochMetrics):
        # Override logging to change the metric label
        print(f"  epoch {epoch:02d} | dice_loss={m.loss:.4f}  mIoU={m.val_map:.4f}")


print("Concrete trainers defined ✓")

Concrete trainers defined ✓


In [13]:
# ── Demo ───────────────────────────────────────────────────────────────────
np.random.seed(1)

print("=== Template Method Demo ===")
print("\n--- Detection (3 epochs) ---")
det_history = DetectionTrainer(model=RFDETRModel()).fit(n_epochs=3)

print("\n--- Segmentation (3 epochs) ---")
seg_history = SegmentationTrainer().fit(n_epochs=3)

print("\n--- Observation ---")
print("Both trainers used the SAME fit() loop from BaseTrainer.")
print("Only train_step() and val_step() were different.")
print(f"Detection  final mAP:  {det_history[-1].val_map:.4f}")
print(f"Segmentation final mIoU: {seg_history[-1].val_map:.4f}")

=== Template Method Demo ===

--- Detection (3 epochs) ---
  [DetectionTrainer] setup — model: RFDETRModel(backbone=dinov2_vitb14, classes=10)


  epoch 01 | loss=0.1189  mAP=0.6812


  epoch 02 | loss=0.1709  mAP=0.6190


  epoch 03 | loss=0.1611  mAP=0.6710

--- Segmentation (3 epochs) ---
  [SegmentationTrainer] setup — Dice loss, mIoU metric


  epoch 01 | dice_loss=0.9999  mIoU=0.7448


  epoch 02 | dice_loss=1.0004  mIoU=0.6969


  epoch 03 | dice_loss=0.9996  mIoU=0.6832

--- Observation ---
Both trainers used the SAME fit() loop from BaseTrainer.
Only train_step() and val_step() were different.
Detection  final mAP:  0.6710
Segmentation final mIoU: 0.6832


---
## Pattern 5 — Adapter

**Intent:** Convert the interface of a class into another interface that clients expect.
The Adapter lets classes work together that couldn't otherwise because of incompatible interfaces.

**ML Problem it solves:**  
Your team has a stable `DetectionModel` interface used everywhere. Third-party backends
(HuggingFace transformers, ONNX Runtime, TensorRT) each have completely different calling
conventions. The Adapter wraps each backend, making it look identical to the rest of your code.

```
Your code                        Adapter                    Third-party backend
──────────                       ───────                    ───────────────────
model.forward(images)  ──────►  HFAdapter.forward()  ────►  pipe(images)
                         ──────►  ORTAdapter.forward() ────►  session.run(None, {"images": ...})
                         ──────►  TRTAdapter.forward() ────►  context.execute_v2([...])
```

In [14]:
# ── 5a: The stable target interface your team code always uses ─────────────

@dataclass
class Detection:
    """Canonical output format — all adapters must produce this."""
    boxes:  np.ndarray   # (N, 4) xyxy float32
    scores: np.ndarray   # (N,)   float32
    labels: np.ndarray   # (N,)   int32

    def __repr__(self):
        return f"Detection(n={len(self.scores)}, top_score={self.scores.max():.3f})"


class InferenceBackend(ABC):
    """
    The target interface — what the rest of the codebase depends on.
    All adapters must implement this.
    """
    @abstractmethod
    def predict(self, image: np.ndarray) -> Detection:
        """Run inference on a single CHW float32 image. Returns canonical Detection."""
        ...

    @property
    @abstractmethod
    def backend_name(self) -> str: ...


print("Target interface defined ✓")

Target interface defined ✓


In [15]:
# ── 5b: Simulate third-party libraries with incompatible interfaces ────────
# In real code these would be: ort.InferenceSession, HF pipeline, trt.IExecutionContext

class _FakeORTSession:
    """Mimics onnxruntime.InferenceSession interface."""
    def run(self, output_names, input_dict):
        # Returns list of numpy arrays: [boxes, scores, labels]
        n = 7
        return [
            np.random.rand(n, 4).astype(np.float32),
            np.random.rand(n).astype(np.float32),
            np.random.randint(0, 5, n).astype(np.int32),
        ]


class _FakeHFPipeline:
    """Mimics a HuggingFace object-detection pipeline interface."""
    def __call__(self, image_array, threshold=0.5):
        # Returns list of dicts with 'box' (dict), 'score', 'label'
        return [
            {"score": round(random.uniform(0.5, 0.99), 3),
             "label": f"class_{random.randint(0,4)}",
             "box": {"xmin": random.randint(0,100), "ymin": random.randint(0,100),
                     "xmax": random.randint(200,400), "ymax": random.randint(200,400)}}
            for _ in range(random.randint(3, 8))
        ]


class _FakeTRTContext:
    """Mimics a TensorRT IExecutionContext interface."""
    def execute_v2(self, bindings: list):
        pass  # writes results in-place to the binding buffers

    class Output:
        boxes  = np.random.rand(5, 4).astype(np.float32)
        scores = np.random.rand(5).astype(np.float32)
        labels = np.random.randint(0, 5, 5).astype(np.int32)

    output = Output()


print("Fake third-party libraries ready ✓")

Fake third-party libraries ready ✓


In [16]:
# ── 5c: Adapter implementations ───────────────────────────────────────────

class ORTAdapter(InferenceBackend):
    """
    Adapts onnxruntime.InferenceSession → InferenceBackend.
    Handles: input naming, dtype casting, output unpacking.
    """
    def __init__(self, ort_session: _FakeORTSession, input_name: str = "images"):
        self._session    = ort_session
        self._input_name = input_name

    @property
    def backend_name(self): return "onnxruntime"

    def predict(self, image: np.ndarray) -> Detection:
        # ORT requires a batch dim and float32
        inp = image[None].astype(np.float32)                   # (1, C, H, W)
        boxes, scores, labels = self._session.run(
            None, {self._input_name: inp}
        )
        # Convert to canonical Detection
        return Detection(boxes=boxes, scores=scores, labels=labels)


class HuggingFaceAdapter(InferenceBackend):
    """
    Adapts a HuggingFace object-detection pipeline → InferenceBackend.
    Handles: dict → numpy conversion, label string → int mapping.
    """
    def __init__(self, hf_pipe: _FakeHFPipeline, score_threshold: float = 0.5):
        self._pipe      = hf_pipe
        self._threshold = score_threshold
        self._label_map: Dict[str, int] = {}   # built lazily

    @property
    def backend_name(self): return "huggingface"

    def predict(self, image: np.ndarray) -> Detection:
        # HF pipeline takes HWC numpy array
        raw = self._pipe(image.transpose(1, 2, 0), threshold=self._threshold)

        boxes, scores, labels = [], [], []
        for det in raw:
            b = det["box"]
            boxes.append([b["xmin"], b["ymin"], b["xmax"], b["ymax"]])
            scores.append(det["score"])
            # Map string label → int lazily
            lbl = det["label"]
            if lbl not in self._label_map:
                self._label_map[lbl] = len(self._label_map)
            labels.append(self._label_map[lbl])

        return Detection(
            boxes  = np.array(boxes,  dtype=np.float32),
            scores = np.array(scores, dtype=np.float32),
            labels = np.array(labels, dtype=np.int32),
        )


class TensorRTAdapter(InferenceBackend):
    """
    Adapts a TensorRT IExecutionContext → InferenceBackend.
    In real code: allocates pinned host buffers, calls execute_v2, copies back.
    """
    def __init__(self, trt_context: _FakeTRTContext):
        self._ctx = trt_context

    @property
    def backend_name(self): return "tensorrt"

    def predict(self, image: np.ndarray) -> Detection:
        # Real impl: copy to GPU buffer, execute, copy results back
        self._ctx.execute_v2(bindings=[image])   # simplified
        out = self._ctx.output
        return Detection(
            boxes  = out.boxes.copy(),
            scores = out.scores.copy(),
            labels = out.labels.copy(),
        )


print("Adapters defined ✓")

Adapters defined ✓


In [17]:
# ── 5d: The evaluation function — works identically regardless of backend ─

def run_evaluation(backend: InferenceBackend, n_images: int = 5) -> dict:
    """
    This function depends ONLY on InferenceBackend.
    It has zero knowledge of ORT, HuggingFace, or TensorRT internals.
    """
    all_scores = []
    for i in range(n_images):
        image = np.random.rand(3, 512, 512).astype(np.float32)
        det   = backend.predict(image)
        all_scores.extend(det.scores.tolist())

    return {
        "backend":     backend.backend_name,
        "n_detections":len(all_scores),
        "mean_score":  float(np.mean(all_scores)),
        "max_score":   float(np.max(all_scores)),
    }


print("=== Adapter Pattern Demo ===")
random.seed(99)
np.random.seed(99)

backends = [
    ORTAdapter(_FakeORTSession()),
    HuggingFaceAdapter(_FakeHFPipeline()),
    TensorRTAdapter(_FakeTRTContext()),
]

print(f"\n{'Backend':<15} {'n_dets':>8} {'mean_score':>12} {'max_score':>10}")
print("-" * 50)
for backend in backends:
    r = run_evaluation(backend, n_images=10)
    print(f"  {r['backend']:<13} {r['n_detections']:>8} {r['mean_score']:>12.4f} {r['max_score']:>10.4f}")

print()
print("run_evaluation() called identically for all three backends.")
print("Swapping backend = swapping one constructor call at the call-site.")

=== Adapter Pattern Demo ===

Backend           n_dets   mean_score  max_score
--------------------------------------------------
  onnxruntime         70       0.4812     0.9973
  huggingface         51       0.7574     0.9790


  tensorrt            50       0.3088     0.6137

run_evaluation() called identically for all three backends.
Swapping backend = swapping one constructor call at the call-site.


In [18]:
# ── 5e: Runtime backend selection via the Registry (combining Patterns 2+5)

backend_registry = Registry("backends")

@backend_registry.register("ort")
class _ORTAdapterFactory:
    @staticmethod
    def build(**kwargs): return ORTAdapter(_FakeORTSession())

@backend_registry.register("hf")
class _HFAdapterFactory:
    @staticmethod
    def build(**kwargs): return HuggingFaceAdapter(_FakeHFPipeline())

@backend_registry.register("trt")
class _TRTAdapterFactory:
    @staticmethod
    def build(**kwargs): return TensorRTAdapter(_FakeTRTContext())


def get_backend_from_config(cfg: dict) -> InferenceBackend:
    factory = backend_registry.build(cfg["backend"])
    return factory.build(**cfg.get("backend_kwargs", {}))


print("=== Adapter + Registry combined ===")
for bname in ["ort", "hf", "trt"]:
    backend = get_backend_from_config({"backend": bname})
    r = run_evaluation(backend, n_images=3)
    print(f"  cfg['backend']={bname!r:5s}  →  {r['backend']:12s}  mean_score={r['mean_score']:.4f}")

=== Adapter + Registry combined ===


  cfg['backend']='ort'  →  onnxruntime   mean_score=0.4562


  cfg['backend']='hf'   →  huggingface   mean_score=0.7352


  cfg['backend']='trt'  →  tensorrt      mean_score=0.3088


---
## Summary

| Pattern | What it fixes | Key mechanism |
|---------|--------------|---------------|
| **Strategy** | if/elif chains for algorithms | Protocol/ABC injected at construction |
| **Factory/Registry** | Hardcoded class imports in orchestration | `@registry.register` decorator + `build(key)` |
| **Observer/Callback** | Side-effects tangled in the training loop | `_fire(event, state)` loop over registered observers |
| **Template Method** | Duplicated training loop boilerplate | Abstract steps in ABC; `fit()` is final/invariant |
| **Adapter** | Incompatible third-party interfaces | Wrapper class translates calls to canonical interface |

### Patterns that work well together
- **Registry + Adapter** → config-driven backend selection (Pattern 5e above)
- **Template Method + Callback** → base loop fires events; subclass overrides steps
- **Strategy + Factory** → register loss functions by name, inject via config

### The guiding question for each MR
> *"If I need to add a new model / loss / backend, how many files do I have to change?"*  
> With these patterns the answer should almost always be **one** — the new class file itself.